**RAPIDS cuML is a GPU-accelerated machine learning library with a scikit-learn-like API that dramatically speeds up training on large datasets.**

# RAPIDS cuML SVC Baseline

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import cudf
from cuml.svm import SVC
from sklearn.model_selection import KFold

import os


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import os
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src.config import Config as Config  
from src.data_loader import load_data, prepare_data

cfg = Config

KAGGLE_EVAL = cfg.KAGGLE_EVAL
RANDOM_STATE = cfg.RANDOM_STATE
TASK = cfg.TASK
USE_POSTPROCESSING = cfg.USE_POSTPROCESSING
TARGET = cfg.TARGET
ID = cfg.ID
SUB_PATH = cfg.SUB_PATH
SUBMIT_PROBABILITIES = cfg.SUBMIT_PROBABILITIES


from src.data_loader import load_data, prepare_data
from src.optuna_utils import run_optuna
from src.evaluation_utils import evaluate_model, evaluate_metric
from src.visualization_utils import plot_feature_importance, plot_learning_curve, shap_summary
from src.postprocessing_utils import optimize_postprocessing, apply_postprocessing
from src.data_splitter import DataSplitter
from src.experiment_tracker import ExperimentTracker
from sklearn.utils import compute_class_weight

# -------------------------------------------------------



sys.path contains: /home/ismail/x42


In [2]:
# -------------------------------
# Load & prepare data
# -------------------------------
X_train, X_test, y_train, y_test = load_data("encoded")
X_train, X_test, y_train_numeric, y_test_numeric, test_ids, num_classes, int_to_label = prepare_data(
    X_train, X_test, y_train, y_test, target=cfg.TARGET, drop_id=True
)

y_train_numeric = np.array(y_train_numeric)
y_test_numeric = np.array(y_test_numeric) if y_test_numeric is not None else None

# from sklearn.model_selection import train_test_split


# -------------------------------
# Data splitter
# -------------------------------
splitter = DataSplitter(
    method="stratified_kfold",
    n_splits=5,
    random_state=RANDOM_STATE,
    folds_path="data/folds.npy"
)
folds = list(splitter.split(X_train, y_train_numeric, reuse_folds=True, verbose=True))


Number of classes: 2
X_train shape: (300000, 116)
X_test shape: (200000, 116)
y_train shape: (300000,)
y_test labels are not available
✅ Loaded 5 folds from data/folds.npy
♻️ Reusing existing folds
--- Splitting data ---
Method: stratified_kfold
Number of splits: 5
Random seeds: [42]
Dataset size: 300000
Total folds: 5

Fold 0: Train size=12000, Val size=3000
Fold 1: Train size=12000, Val size=3000
Fold 2: Train size=12000, Val size=3000
Fold 3: Train size=12000, Val size=3000
Fold 4: Train size=12000, Val size=3000


In [3]:
import cupy as cp
import gc

# Clear GPU memory
cp.get_default_pinned_memory_pool().free_all_blocks()
cp.get_default_memory_pool().free_all_blocks()
gc.collect()

11

In [4]:
import numpy as np
import pandas as pd
import gc
import time
import warnings

# RAPIDS / GPU
import cudf
import cupy as cp
from cuml.svm import SVC

# Scikit-learn / Metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

# ------------------- UTILS ------------------------------------
def clear_gpu_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

# ------------------- DATA LOADING -----------------------------
# RUN THIS FIRST: Ensure your data loading cell is actually executed!
# X_train, X_test, y_train, y_test, y_train_numeric, num_classes etc. must exist.

# ------------------- GPU CONVERSION ---------------------------
print("Moving data to GPU...")
X_gpu = cudf.DataFrame(X_train).astype('float32')
y_gpu = cudf.Series(y_train_numeric)
X_test_gpu = cudf.DataFrame(X_test).astype('float32')

# ------------------- K-FOLD CONFIG ----------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Containers for predictions
oof_preds = np.zeros(len(y_train_numeric)) if cfg.TASK.lower() == "binary" else np.zeros((len(y_train_numeric), num_classes))
test_preds = np.zeros(len(X_test)) if cfg.TASK.lower() == "binary" else np.zeros((len(X_test), num_classes))

# ------------------- THE LOOP ---------------------------------
print(f"Starting {N_SPLITS}-Fold Cross-Validation...")

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train_numeric)):
    fold_start = time.time()
    
    # 1. Subset the GPU data
    X_tr, X_val = X_gpu.iloc[train_idx], X_gpu.iloc[valid_idx]
    y_tr, y_val = y_gpu.iloc[train_idx], y_gpu.iloc[valid_idx]

    # 2. Initialize SVC 
    # NOTE: probability=True is the most common cause of crashes. 
    # cache_size=1000 keeps the GPU from eating all VRAM for kernel caching.
    model = SVC(
        C=1.0, 
        kernel='rbf', 
        probability=True, 
        cache_size=1000, 
        max_iter=5000  # Prevents long hangs
    )

    # 3. Fit & Predict
    model.fit(X_tr, y_tr)
    
    val_probs = model.predict_proba(X_val).to_numpy()
    fold_test_probs = model.predict_proba(X_test_gpu).to_numpy()

    # 4. Store Preds
    if cfg.TASK.lower() == "binary":
        oof_preds[valid_idx] = val_probs[:, 1]
        test_preds += fold_test_probs[:, 1] / N_SPLITS
        fold_score = roc_auc_score(y_val.to_numpy(), val_probs[:, 1])
    else:
        oof_preds[valid_idx] = val_probs
        test_preds += fold_test_probs / N_SPLITS
        fold_score = roc_auc_score(y_val.to_numpy(), val_probs, multi_class="ovr")

    print(f"Fold {fold+1} | Score: {fold_score:.5f} | Time: {time.time() - fold_start:.2f}s")

    # 5. CRITICAL CLEANUP
    del model, X_tr, X_val, y_tr, y_val
    clear_gpu_memory()

# ------------------- FINAL OOF SCORE --------------------------
if cfg.TASK.lower() == "binary":
    final_score = roc_auc_score(y_train_numeric, oof_preds)
else:
    final_score = roc_auc_score(y_train_numeric, oof_preds, multi_class="ovr")

print(f"\nFinal OOF AUC: {final_score:.5f}")

Moving data to GPU...


: 